# Meta Model - Ensembling Five Models using Stacking


## Imports

In [32]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [33]:
RANDOM_STATE = 42
target_col = 'Depression'

## Load Base Models

In [34]:
dt_model = joblib.load("/workspaces/CS_Group07_StudentDepressionDetection/notebooks/dunith_decision_tree/decision_tree_model.joblib")
rf_model = joblib.load("/workspaces/CS_Group07_StudentDepressionDetection/notebooks/fc211009_Themiya_random_forrest/random_forest_student_depression.joblib")
svm_model = joblib.load("/workspaces/CS_Group07_StudentDepressionDetection/notebooks/fc211011_kaveesha_svm_model/best_svm_model.joblib")
gb_model = joblib.load("/workspaces/CS_Group07_StudentDepressionDetection/notebooks/menura-gradient_boosting_classifier/gbc_model.joblib")
lr_model = joblib.load("/workspaces/CS_Group07_StudentDepressionDetection/notebooks/Rushani_Logistic_Regression/final_logistic_model_clean.joblib")

models = {
    'DecisionTree': dt_model,
    'RandomForest': rf_model,
    'SVM': svm_model,
    'GradientBoosting': gb_model,
    'LogisticRegression': lr_model
}

## Load Dataset 

In [35]:
df = pd.read_csv("/workspaces/CS_Group07_StudentDepressionDetection/notebooks/menura-gradient_boosting_classifier/preprocessed_student_depression(GBC-model).csv")  
X = df.drop(columns=[target_col])
y = df[target_col]

## Split Dataset

In [36]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

## Feature Engineering 

### Identify feature types

In [37]:
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

cols_to_scale = [col for col in numerical_cols if col != target_col]


### Scale Numerical Features

In [38]:
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train_scaled[cols_to_scale])
X_test_scaled[cols_to_scale] = scaler.transform(X_test_scaled[cols_to_scale])


### One-Hot Encode Categorical Features

In [39]:
encoder = OneHotEncoder(handle_unknown='ignore', drop='first')

X_train_cat = encoder.fit_transform(X_train_scaled[categorical_cols])
X_test_cat = encoder.transform(X_test_scaled[categorical_cols])

# Convert to DataFrames
encoded_col_names = encoder.get_feature_names_out(categorical_cols)
X_train_cat_df = pd.DataFrame(X_train_cat.toarray(), columns=encoded_col_names, index=X_train_scaled.index)
X_test_cat_df = pd.DataFrame(X_test_cat.toarray(), columns=encoded_col_names, index=X_test_scaled.index)


## Combining

In [40]:
X_train_final = pd.concat([X_train_scaled[cols_to_scale], X_train_cat_df], axis=1)
X_test_final = pd.concat([X_test_scaled[cols_to_scale], X_test_cat_df], axis=1)

print("Train shape:", X_train_final.shape)
print("Test shape:", X_test_final.shape)

Train shape: (22293, 46)
Test shape: (5574, 46)


## Generate Meta Features

In [41]:
meta_train = pd.DataFrame()
meta_test = pd.DataFrame()

for name, model in models.items():
    print(f"Generating predictions from {name}...")
    try:
        meta_train[name] = model.predict_proba(X_train_final)[:, 1]
        meta_test[name] = model.predict_proba(X_test_final)[:, 1]
    except Exception as e:
        print(f"⚠️ {name} failed predict_proba, using predict() instead: {e}")
        meta_train[name] = model.predict(X_train_final)
        meta_test[name] = model.predict(X_test_final)


Generating predictions from DecisionTree...
Generating predictions from RandomForest...
Generating predictions from SVM...


/opt/conda/envs/ml-env/lib/python3.11/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but SVC was fitted without feature names
  warnings.warn(


/opt/conda/envs/ml-env/lib/python3.11/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but SVC was fitted without feature names
  warnings.warn(


Generating predictions from GradientBoosting...
Generating predictions from LogisticRegression...
⚠️ LogisticRegression failed predict_proba, using predict() instead: X has 46 features, but LogisticRegression is expecting 38 features as input.


/opt/conda/envs/ml-env/lib/python3.11/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/opt/conda/envs/ml-env/lib/python3.11/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


ValueError: X has 46 features, but LogisticRegression is expecting 38 features as input.